In [ ]:
import unittest
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import sys
import os

# Ensure glow is in path
if '..' not in sys.path:
    sys.path.append('..')

from glow.analysis.positive import get_channel_data_for_samples, calculate_positive_stats
from glow.analysis.stats import get_summary_statistics

class MockExperiment:
    def __init__(self, data):
        self.data = data
    @property
    def conditions(self):
        # Simulate the crash
        raise KeyError("Metadata missing")

class TestHistogramIntergration(unittest.TestCase):
    def setUp(self):
        # Create a dataframe with mixed sample types (some might read as int, some string?)
        # Standard flow data often has 'Sample' as a string column
        self.df = pd.DataFrame({
            'Sample': ['S1', 'S1', 'S2', 'S2', 'S3'],
            'GFP-A': [10.0, 100.0, 1000.0, 10000.0, -50.0], # Mixed scales and negative
            'cells': [True, True, True, True, True],
            'singlets': [True, True, True, True, True]
        })
        self.experiment = MockExperiment(self.df)

    def test_end_to_end_data_flow(self):
        """Simulate the flow from Stats generation -> UI Selection -> Data Retrieval"""
        
        # 1. UI: Get Summary Stats to populate sample list
        stats_df = get_summary_statistics(self.experiment)
        sample_list = stats_df['Sample'].unique().tolist()
        
        print(f"UI Sample List: {sample_list}")
        
        # Verify samples are found
        self.assertIn('S1', sample_list)
        self.assertIn('S3', sample_list)
        
        # 2. UI: User selects 'S2' (High value) and 'S3' (Negative value)
        selected_samples = ['S2', 'S3']
        curr_channel = 'GFP-A'
        
        # 3. UI: Fetch Data
        data_map = get_channel_data_for_samples(self.experiment, curr_channel, selected_samples)
        
        # Verify Data Map
        self.assertIn('S2', data_map)
        self.assertIn('S3', data_map)
        self.assertNotIn('S1', data_map) # Should not retrieve unselected
        
        # Check values
        s2_vals = data_map['S2']
        s3_vals = data_map['S3']
        
        print(f"Data S2: {s2_vals}")
        print(f"Data S3: {s3_vals}")
        
        self.assertTrue(len(s2_vals) == 2)
        self.assertTrue(len(s3_vals) == 1)
        
        # 4. UI: Logic for Min/Max and Log Filtering
        all_values = []
        for s in data_map:
            all_values.extend(data_map[s])
            
        raw_min = float(np.min(all_values))
        raw_max = float(np.max(all_values))
        
        print(f"Raw Range: {raw_min} to {raw_max}")
        self.assertEqual(raw_min, -50.0)
        self.assertEqual(raw_max, 10000.0)
        
        # Log Logic
        use_log = True
        pos_vals = [v for v in all_values if v > 0]
        
        if not pos_vals:
            use_log = False
            
        valid_min = float(np.min(pos_vals)) if pos_vals else raw_min
        valid_max = float(np.max(pos_vals)) if pos_vals else raw_max
        
        print(f"Log Range: {valid_min} to {valid_max}")
        self.assertEqual(valid_min, 1000.0)
        
        # 5. Plotly Object Creation (Smoke Test)
        fig = go.Figure()
        for s in selected_samples:
            vals = data_map[s]
            if use_log:
                vals = vals[vals > 0]
            
            if len(vals) > 0:
                fig.add_trace(go.Histogram(x=vals, name=s))
                
        # If S3 was all negative, it should not be in the plot if log is True
        # checking trace names
        trace_names = [t.name for t in fig.data]
        self.assertIn('S2', trace_names)
        self.assertNotIn('S3', trace_names) # S3 (-50) filtered out by log logic
        
        print("Test Passed: Workflow verified.")

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


In [11]:
import os
import shutil
import sys

# Ensure local modules can be imported
sys.path.append(os.getcwd())

from active_learning.learner import ActiveLearner
from active_learning.storage import ActiveLearningLogger

# 1. Setup Test Environment
TEST_STORAGE_DIR = "test_active_learning_data"
TEST_MODEL_FILE = "test_model_weights.json"

# Clean up previous runs
if os.path.exists(TEST_STORAGE_DIR):
    shutil.rmtree(TEST_STORAGE_DIR)
os.makedirs(TEST_STORAGE_DIR, exist_ok=True)

print("Test environment setup complete.")

Test environment setup complete.


In [12]:
# 2. Simulate User Feedback Generation

logger = ActiveLearningLogger(storage_dir=TEST_STORAGE_DIR)

# Scenario: User gates CellLineA
# Live Gate (5 vertices)
logger.log_gate_feedback(
    sample_name="Sample_A1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="SSC-A",
    gate_type="Polygon",
    gate_params={'vertices': [[1, 1], [2, 1], [3, 2], [2, 3], [1, 2]]}, # 5 points = Live
    user_decision="accepted"
)

# Singlet Gate (4 vertices)
logger.log_gate_feedback(
    sample_name="Sample_A1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="FSC-H",
    gate_type="Polygon",
    gate_params={'vertices': [[10, 10], [20, 10], [20, 20], [10, 20]]}, # 4 points = Singlet
    user_decision="accepted"
)

# Scenario: User gates CellLineB (Different cell line)
# Live Gate (5 vertices)
logger.log_gate_feedback(
    sample_name="Sample_B1",
    cell_line=" CellLineB ", # Note extra spaces to test normalization
    x_channel="FSC-A",
    y_channel="SSC-A",
    gate_type="Polygon",
    gate_params={'vertices': [[5, 5], [6, 5], [7, 6], [6, 7], [5, 6]]},
    user_decision="manual_override"
)

print("Simulated user feedback logged.")

Simulated user feedback logged.


In [9]:
# 3. specific Active Learner Training

learner = ActiveLearner(storage_dir=TEST_STORAGE_DIR, model_filename=TEST_MODEL_FILE)

# Initial state should be empty or default
print("Known cell lines before training:", learner.get_known_cell_lines())

# Train (Process logs)
learner.train()

print("Known cell lines after training:", learner.get_known_cell_lines())

# Verify Predictions
# CellLineA
pred_live_a = learner.predict("CellLineA", "Live")
pred_singlet_a = learner.predict("CellLineA", "Singlet")

print(f"\nCellLineA Live Gate (Expected 5 pts): {len(pred_live_a) if pred_live_a else 'None'}")
print(f"CellLineA Singlet Gate (Expected 4 pts): {len(pred_singlet_a) if pred_singlet_a else 'None'}")

# CellLineB (Should be normalized)
pred_live_b = learner.predict("CellLineB", "Live")
print(f"CellLineB Live Gate (Expected 5 pts): {len(pred_live_b) if pred_live_b else 'None'}")

# CellLineC (Unknown)
pred_live_c = learner.predict("CellLineC", "Live")
print(f"CellLineC Live Gate (Expected None): {pred_live_c}")

No existing model weights found. Starting fresh.
Known cell lines before training: []
Training complete. Knowledge base updated for 2 cell lines.
Model weights saved to test_active_learning_data/test_model_weights.json
Known cell lines after training: ['CellLineA', 'CellLineB']

CellLineA Live Gate (Expected 5 pts): 5
CellLineA Singlet Gate (Expected 4 pts): 4
CellLineB Live Gate (Expected 5 pts): 5
CellLineC Live Gate (Expected None): None


In [10]:
# 4. Persistence Test (Load from disk)

# Create a new instance to simulate restarting the application
learner_new = ActiveLearner(storage_dir=TEST_STORAGE_DIR, model_filename=TEST_MODEL_FILE)

print("Loaded known cell lines from disk:", learner_new.get_known_cell_lines())

assert "CellLineA" in learner_new.get_known_cell_lines()
assert "CellLineB" in learner_new.get_known_cell_lines()

print("\nTest passed successfully!")

Model weights loaded from test_active_learning_data/test_model_weights.json
Loaded known cell lines from disk: ['CellLineA', 'CellLineB']

Test passed successfully!
